### A small notebook to test functions from the simulation directory

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from aml_working_paper.simulation.states import sample_next_state
from scipy.special import softmax
from aml_working_paper.simulation.equations import generate_economic_context, simulate_account, simulate_accounts

In [5]:
BASE_TRANSITION = np.array(
    [
    [0.94, 0.05, 0.01],
    [0.10, 0.75, 0.15],
    [0.03, 0.17, 0.80],
    ],
    dtype=float,
)


In [13]:
NETWORK_EXPOSURE_EFFECT = np.array([-1.0, 0.25, 1.25], dtype=float)

In [20]:
rng = np.random.default_rng(42)

state = 0

for _ in range(20):
    state = sample_next_state(
        previous_state=state,
        account_exposure=0.5,
        rng=rng,
    )

    print(state)

0
0
0
0
0
2
2
2
2
2
2
2
2
2
2
2
2
1
2
2


In [ ]:
probabilities = np.array([0.94, 0.05, 0.01])

logits = np.log(probabilities)

recovered = softmax(logits)

print(logits)
print(recovered)

In [ ]:
account_exposure_EFFECT = np.array(
    [-1.0, 0.25, 1.25],
    dtype=float,
)

In [ ]:
def test_generate_economic_context_returns_float():
    rng = np.random.default_rng(42)

    context = generate_economic_context(
        previous_context=0.0,
        rng=rng,
    )

    assert isinstance(context, float)

In [ ]:
test_generate_economic_context_returns_float()

In [ ]:
rng = np.random.default_rng(42)

economic_context = 0.0

for t in range(10):
    economic_context = generate_economic_context(
        previous_context=economic_context,
        rng=rng,
    )

    print(t, economic_context)

In [ ]:
import numpy as np

from aml_working_paper.simulation.equations import (
    generate_economic_context,
    generate_account_exposure,
)

from aml_working_paper.simulation.states import sample_next_state


rng = np.random.default_rng(42)

economic_context = 0.0
previous_aml_state = 0

for t in range(20):
    economic_context = generate_economic_context(
        previous_context=economic_context,
        rng=rng,
    )

    account_exposure = generate_account_exposure(
        economic_context=economic_context,
        rng=rng,
    )

    aml_state = sample_next_state(
        previous_state=previous_aml_state,
        account_exposure=account_exposure,
        rng=rng,
    )

    print(
        t,
        round(economic_context, 3),
        round(account_exposure, 3),
        aml_state,
    )

    previous_aml_state = aml_state

In [1]:
import numpy as np

seed_sequence = np.random.SeedSequence(42)

child_seeds = seed_sequence.spawn(6)

rng_economic = np.random.default_rng(child_seeds[0])
rng_network = np.random.default_rng(child_seeds[1])
rng_state = np.random.default_rng(child_seeds[2])
rng_structuring = np.random.default_rng(child_seeds[3])
rng_cross_border = np.random.default_rng(child_seeds[4])
rng_volume = np.random.default_rng(child_seeds[5])

In [2]:
rng_structuring

Generator(PCG64) at 0x7EF37811E7A0

In [4]:
noise = rng_structuring.normal(0, 0.6)
structuring = -2 + 1.3 * aml_state + noise

NameError: name 'aml_state' is not defined

In [2]:
rng_state = np.random.default_rng(child_seeds[2])

In [2]:
df = simulate_account(
    periods=100_000,
    seed=42,
)

In [ ]:
correlation = df[
    ["economic_context", "account_exposure"]
].corr()

print(correlation)

                  economic_context  network_exposure
economic_context          1.000000          0.643373
network_exposure          0.643373          1.000000


In [ ]:
print(
    "Context/exposure correlation:",
    df["economic_context"].corr(
        df["account_exposure"]
    ),
)

print(
    "\nExposure by economic context quintile:"
)

df["economic_context_bin"] = pd.qcut(
    df["economic_context"],
    q=5,
    labels=False,
)

print(
    df.groupby("economic_context_bin")[
        "account_exposure"
    ].mean()
)

print(
    "\nStructuring by AML state:"
)

print(
    df.groupby("aml_state")[
        "structuring_score"
    ].mean()
)

print(
    "\nCross-border by AML state:"
)

print(
    df.groupby("aml_state")[
        "cross_border_ratio"
    ].mean()
)
print("transaction volumne mean",
    df.groupby("aml_state")[
        "transaction_volume"
    ].mean()
)
print("transaction volumne median",
    df.groupby("aml_state")[
        "transaction_volume"
    ].median()
)

Context/exposure correlation: 0.6433731142770676

Exposure by economic context quintile:
economic_context_bin
0    0.063371
1    0.102461
2    0.138409
3    0.183956
4    0.279785
Name: network_exposure, dtype: float64

Structuring by AML state:
aml_state
0    0.132701
1    0.342903
2    0.635849
Name: structuring_score, dtype: float64

Cross-border by AML state:
aml_state
0    0.105611
1    0.204721
2    0.353581
Name: cross_border_ratio, dtype: float64
transaction volumne mean aml_state
0    13.927050
1    19.592374
2    29.033594
Name: transaction_volume, dtype: float64
transaction volumne median aml_state
0    13.007875
1    18.232564
2    27.059376
Name: transaction_volume, dtype: float64


In [ ]:
from_one = df[
    df["previous_aml_state"] == 1
].copy()

from_one["exposure_bin"] = pd.qcut(
    from_one["account_exposure"],
    q=5,
    labels=False,
)

one_to_active = (
    from_one.groupby("exposure_bin")["aml_state"]
    .apply(lambda x: (x == 2).mean())
)

print(
    "\nP(Z_t = 2 | Z_t-1 = 1):"
)
print(one_to_active)


P(Z_t = 2 | Z_t-1 = 1):
exposure_bin
0    0.154963
1    0.160617
2    0.161677
3    0.178403
4    0.209399
Name: aml_state, dtype: float64


In [6]:
from_zero = df[
    df["previous_aml_state"] == 0
].copy()

In [ ]:
from_zero["exposure_bin"] = pd.qcut(
    from_zero["account_exposure"],
    q=5,
    labels=False,
)

In [8]:
escalation_probability = (
    from_zero.groupby("exposure_bin")["aml_state"]
    .apply(lambda x: (x >= 1).mean())
)

print(
    "P(Z_t >= 1 | Z_t-1 = 0):"
)
print(escalation_probability)

P(Z_t >= 1 | Z_t-1 = 0):
exposure_bin
0    0.062929
1    0.062455
2    0.066072
3    0.078490
4    0.093079
Name: aml_state, dtype: float64


In [2]:
panel = simulate_accounts(
    n_accounts=5,
    periods=20,
    seed=42,
)

In [3]:
panel

,account_id,time,economic_context,network_exposure,previous_aml_state,aml_state,structuring_score,cross_border_ratio,transaction_volume
0,0,0,0.529419,0.062469,0,0,0.097453,0.060671,14.753837
1,0,1,0.550935,0.183982,0,0,0.257241,0.058192,17.693912
2,0,2,0.527093,0.247254,0,0,0.069465,0.171748,13.516781
3,0,3,-0.991129,0.117232,0,0,0.247049,0.077314,10.770161
4,0,4,-0.879904,0.025602,0,0,0.078729,0.062309,10.318240
...,...,...,...,...,...,...,...,...,...
95,4,15,0.600839,0.195095,0,0,0.116554,0.496430,26.928862
96,4,16,0.067812,0.062714,0,0,0.162478,0.042993,21.620991
97,4,17,-0.366724,0.130598,0,0,0.056983,0.080383,9.885028
98,4,18,-0.985434,0.058170,0,0,0.084980,0.039444,14.079250


In [7]:
panel.head(15)

,account_id,time,economic_context,network_exposure,previous_aml_state,aml_state,structuring_score,cross_border_ratio,transaction_volume
0,0,0,0.529419,0.062469,0,0,0.097453,0.060671,14.753837
1,0,1,0.550935,0.183982,0,0,0.257241,0.058192,17.693912
2,0,2,0.527093,0.247254,0,0,0.069465,0.171748,13.516781
3,0,3,-0.991129,0.117232,0,0,0.247049,0.077314,10.770161
4,0,4,-0.879904,0.025602,0,0,0.078729,0.062309,10.318240
5,0,5,-0.847161,0.061228,0,0,0.195600,0.065137,10.393637
6,0,6,-0.138784,0.129359,0,0,0.043523,0.062539,13.598796
7,0,7,-0.113399,0.117791,0,0,0.142606,0.098772,9.249842
8,0,8,0.146166,0.129983,0,0,0.086726,0.093636,18.365306
9,0,9,0.344113,0.171583,0,0,0.098575,0.110352,10.737278


In [5]:
print(
    panel.groupby("time")[
        "economic_context"
    ].nunique()
)

time
0     1
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
Name: economic_context, dtype: int64


In [8]:
panel_1 = simulate_accounts(
    n_accounts=5,
    periods=20,
    seed=42,
)

panel_2 = simulate_accounts(
    n_accounts=5,
    periods=20,
    seed=42,
)

pd.testing.assert_frame_equal(
    panel_1,
    panel_2,
)

In [ ]:
print(
    panel.groupby("time")[
        "account_exposure"
    ].nunique()
)

time
0     5
1     5
2     5
3     5
4     5
5     5
6     5
7     5
8     5
9     5
10    5
11    5
12    5
13    5
14    5
15    5
16    5
17    5
18    5
19    5
Name: network_exposure, dtype: int64


In [10]:
assert len(panel) == 5 * 20

In [11]:
print(
    panel.groupby("aml_state")[
        [
            "structuring_score",
            "cross_border_ratio",
            "transaction_volume",
        ]
    ].mean()
)

           structuring_score  cross_border_ratio  transaction_volume
aml_state                                                           
0                   0.138136            0.119316           15.762131
1                   0.345227            0.179935           20.318639
2                   0.646783            0.417410           29.244222
